# Dataset augmentation for before/after renovation pairs

Expands 96 cleaned before/after training pairs into the augmented LoRA set: each pair becomes the original plus 5 variants.

**Design**
- Geometric transforms (crop-zoom, flip, small rotation) use one random state per (pair, variant), applied **identically** to before and after — otherwise the model would learn "renovation = mirror the room".
- Photometric degradation (jitter, grain, JPEG) is applied to the **before** side only, matching production, where the user uploads a noisy phone photo and expects a clean result.

Holdout (10 pairs) is left un-augmented and is not touched here.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Load labelled pairs (train split)

In [3]:
import os
from pathlib import Path
import pandas as pd

BASE = '/content/drive/MyDrive/lora_dataset_final'
CSV = f'{BASE}/files_for_vector_index/pairs_with_style.csv'

df = pd.read_csv(CSV)

index = {}
for root, _, files in os.walk(BASE):
    parent = os.path.basename(root)
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            index[(parent, f)] = os.path.join(root, f)

df['p_before'] = df['src_before'].apply(lambda p: index.get(('before', Path(p).name)))
df['p_after']  = df['src_after'].apply(lambda p: index.get(('after',  Path(p).name)))

train = df[(df['split'] == 'train') & df['p_before'].notna() & df['p_after'].notna()]
print(f"train pairs ready: {len(train)}/96")

train pairs ready: 96/96


## 3. Paired augmentation pipeline

In [ ]:
import io, os, random
from pathlib import Path
import numpy as np
from PIL import Image, ImageEnhance

NUM_VARIANTS = 5
ZOOM_RANGE   = (0.90, 0.97)
ROTATION_DEG = 3.0
JITTER_RANGE = (0.92, 1.08)
NOISE_SIGMA  = (2.0, 6.0)
JPEG_QUALITY = (72, 92)
SEED         = 42


def make_params(pair_id, version):
    """One random state per (pair, version) — never per side."""
    rng = random.Random(f"{SEED}:{pair_id}:{version}")
    return {
        "zoom":       rng.uniform(*ZOOM_RANGE),
        "crop_x":     rng.random(),
        "crop_y":     rng.random(),
        "flip":       rng.random() < 0.5,
        "rotate":     rng.uniform(-ROTATION_DEG, ROTATION_DEG),
        "brightness": rng.uniform(*JITTER_RANGE),
        "contrast":   rng.uniform(*JITTER_RANGE),
        "saturation": rng.uniform(*JITTER_RANGE),
        "noise_sigma": rng.uniform(*NOISE_SIGMA),
        "noise_seed":  rng.randrange(2**32),
        "jpeg":        rng.randint(*JPEG_QUALITY),
    }


def geometric(img, p):
    """Crop-zoom, optional flip, small rotation. Identical for both sides."""
    w, h = img.size
    cw, ch = int(w * p["zoom"]), int(h * p["zoom"])
    left = int((w - cw) * p["crop_x"])
    top  = int((h - ch) * p["crop_y"])
    img = img.crop((left, top, left + cw, top + ch)).resize((w, h), Image.LANCZOS)

    if p["flip"]:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)

    if abs(p["rotate"]) > 0.01:
        img = img.rotate(p["rotate"], resample=Image.BICUBIC,
                         expand=False, fillcolor=(255, 255, 255))
    return img


def photometric(img, p):
    """Jitter, grain, JPEG recompression. BEFORE side only."""
    for enhancer, key in ((ImageEnhance.Brightness, "brightness"),
                          (ImageEnhance.Contrast,   "contrast"),
                          (ImageEnhance.Color,      "saturation")):
        img = enhancer(img).enhance(p[key])

    arr = np.asarray(img, dtype=np.float32)
    arr += np.random.default_rng(p["noise_seed"]).normal(0, p["noise_sigma"], arr.shape)
    img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=p["jpeg"])
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def augment_pair(before_path, after_path, out_before, out_after, pair_id, stem):
    """Write orig + NUM_VARIANTS versions of one pair."""
    before = Image.open(before_path).convert("RGB")
    after  = Image.open(after_path).convert("RGB")

    os.makedirs(out_before, exist_ok=True)
    os.makedirs(out_after,  exist_ok=True)

    before.save(f"{out_before}/{stem}_orig.jpg", quality=95)
    after.save( f"{out_after}/{stem}_orig.jpg",  quality=95)

    for v in range(NUM_VARIANTS):
        p = make_params(pair_id, v)
        b = photometric(geometric(before, p), p)
        a = geometric(after, p)
        b.save(f"{out_before}/{stem}_v{v}.jpg", quality=95)
        a.save(f"{out_after}/{stem}_v{v}.jpg",  quality=95)